# Imports and setup

In [18]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler
from transformers import get_linear_schedule_with_warmup

# Hardware optimization for A100
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Custom modules
import dataset
import model_utils
import preprocess
import eval as ev
import validate as val

# Reload modules if needed (useful in notebooks)
import importlib
importlib.reload(dataset)
importlib.reload(model_utils)
importlib.reload(ev)
importlib.reload(val)

# Specific classes/functions for convenience
from dataset import TicketDataset
from model_utils import MultiTaskXLMR, train_epoch, train_epoch_multilingual
from transformers import AutoTokenizer

In [2]:
# --- 4. Initialize Components ---
NUM_EPOCHS = 3  # Setting to 3 epochs
model_name = "xlm-roberta-large"
BATCH_SIZE = 64

# Data Preprocessing

In [3]:
print("===== Preprocessing Jigsaw English Dataset =====")
preprocess.preprocess_jigsaw()

print("\n===== Preprocessing Multilingual Toxic Dataset =====")
preprocess.preprocess_multilingual()


===== Preprocessing Jigsaw English Dataset =====
Loading English Jigsaw dataset...


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/68.8M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/60.4M [00:00<?, ?B/s]

test_labels.csv: 0.00B [00:00, ?B/s]

...

Generating hin split:   0%|          | 0/4363 [00:00<?, ? examples/s]

Generating tt split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating ja split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Merged non-English multilingual dataset saved:
Train: 53099 rows → data/processed/multilingual_toxic/merged_non_en_train.csv
Test: 13275 rows → data/processed/multilingual_toxic/merged_non_en_test.csv


(                                                    text  toxic language
 51737                                 שלא תצחק כמו יונית      0       he
 11960  Das sind strategische Punkte....das geschieht ...      0       de
 32400  عند الدقيقة 3:53 التي تتكلم عن مؤسس شركة netsc...      0       ar
 53421  Vishnu Ji ka naya awtar   papiyo ko mitane  ke...      0      hin
 10666  es geht hier nicht darum irgendwelche Leute zu...      0       de
 ...                                                  ...    ...      ...
 2992   этот пидор просто кипятком ссыт... бобик сдох ...      1       ru
 59975  мин монда берни дә аңламыйм, шуңа күрә бу мәгъ...      0       tt
 59670  туган көнең белән,сәламәтлек,зур бәхет, иң якт...      0       tt
 13513  Die israelische Polizei sieht zu, wie Juden in...      1       de
 31134  امين يا رب ولك بالمثل يا نهى 😍 ونتمنى لك كل خي...      0       ar
 
 [53099 rows x 3 columns],
                                                     text  toxic language
 13980  H

In [4]:
# --- 3. Load and Prepare Data ---
train_df = pd.read_csv("data/processed/jigsaw/jigsaw_train.csv").fillna("")
intent_cols = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

train_labels = [
    {'toxic': t, 'intents': i.tolist()}
    for t, i in zip(train_df['toxic'].values, train_df[intent_cols].values)
]

## Model Training (On only Jigsaw dataset)

In [5]:
# Train the model
train_ds = TicketDataset(train_df['comment_text'].values, train_labels, model_name=model_name)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

model = MultiTaskXLMR(model_name=model_name, num_intents=len(intent_cols)).to(device)

# Using a slightly lower LR for 'Large' model stability over multiple epochs
optimizer = AdamW(model.parameters(), lr=1e-5, fused=True)
scaler = GradScaler()

# Update total_steps to include all epochs
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1), # 10% warmup is standard
    num_training_steps=total_steps
)

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipython-input-2724181050.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [6]:
# --- 5. Execute Training Loop ---
print(f"Starting Multi-Task Training on {device} for {NUM_EPOCHS} epochs...")

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch + 1}/{NUM_EPOCHS} ---")

    # The train_epoch function we wrote earlier handles the tqdm bar and live loss
    avg_loss = train_epoch(model, train_loader, optimizer, device, scaler, scheduler)

    print(f"Epoch {epoch + 1} Complete. Average Joint Loss: {avg_loss:.4f}")

    # Save a checkpoint after every epoch
    checkpoint_path = f"xlmr_large_multitask_epoch{epoch+1}.bin"
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Saved checkpoint: {checkpoint_path}")

print("\nTraining Finished!")

Starting Multi-Task Training on cuda for 3 epochs...

--- Epoch 1/3 ---


Epoch 0:   0%|          | 0/2244 [00:00<?, ?it/s]/content/model_utils.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
Epoch 0: 100%|██████████| 2244/2244 [07:16<00:00,  5.14it/s, loss=0.0470]


Epoch 1 Complete. Average Joint Loss: 0.1640
Saved checkpoint: xlmr_large_multitask_epoch1.bin

--- Epoch 2/3 ---


Epoch 0: 100%|██████████| 2244/2244 [07:15<00:00,  5.15it/s, loss=0.0811]


Epoch 2 Complete. Average Joint Loss: 0.0882
Saved checkpoint: xlmr_large_multitask_epoch2.bin

--- Epoch 3/3 ---


Epoch 0: 100%|██████████| 2244/2244 [07:15<00:00,  5.15it/s, loss=0.1367]


Epoch 3 Complete. Average Joint Loss: 0.0696
Saved checkpoint: xlmr_large_multitask_epoch3.bin

Training Finished!


# **Validating** the model multilingual data

In [14]:
# 1. Load Multilingual Test Data
test_df = pd.read_csv("data/processed/multilingual_toxic/merged_non_en_test.csv").fillna("")

# 2. Create Dataset
# We use [0]*5 for intents because this test set only evaluates toxicity
test_labels = [{'toxic': t, 'intents': [0]*5} for t in test_df['toxic'].values]
test_ds = TicketDataset(test_df['text'].values, test_labels, model_name="xlm-roberta-large")
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

# 3. Load the Checkpoint
# Note: Use your final multilingual-tuned checkpoint here for best results!
checkpoint_path = "xlmr_large_multitask_epoch3.bin"
model.load_state_dict(torch.load(checkpoint_path))
model.to(device)
model.eval()

# 4. Run Evaluation (Handling the "Too many values to unpack" error)
# The *extra_metrics catches any additional values your function returns
results = ev.run_evaluation(
    model,
    test_loader,
    device,
    lang_list=test_df['language'].values
)

# Unpack based on what the function actually returns
report, auc, lang_f1s = results[0], results[1], results[2]

print("\n--- OVERALL MULTILINGUAL REPORT ---")
print(report)
print(f"Global AUC-ROC: {auc:.4f}")

print("\n--- PER-LANGUAGE F1 SCORES ---")
# If lang_f1s is a dictionary, this will print it beautifully
for lang, score in lang_f1s.items():
    print(f"{lang.upper():<10}: {score:.4f}")

Evaluating: 100%|██████████| 208/208 [00:13<00:00, 15.05it/s]



--- OVERALL MULTILINGUAL REPORT ---
              precision    recall  f1-score   support

       Clean       0.63      0.95      0.76      6674
       Toxic       0.90      0.45      0.60      6601

    accuracy                           0.70     13275
   macro avg       0.77      0.70      0.68     13275
weighted avg       0.76      0.70      0.68     13275

Global AUC-ROC: 0.8511

--- PER-LANGUAGE F1 SCORES ---
DE        : 0.6298
AM        : 0.5672
RU        : 0.9269
HIN       : 0.5146
JA        : 0.5488
IT        : 0.7833
ZH        : 0.4770
FR        : 0.9396
AR        : 0.6137
UK        : 0.6854
TT        : 0.6302
ES        : 0.7691
HE        : 0.7751
HI        : 0.5093


## Multilingual Model training

In [15]:
# Load your multilingual data
multi_train_df = pd.read_csv("data/processed/multilingual_toxic/merged_non_en_train.csv").fillna("")

# Note: If your multilingual CSV doesn't have the 5 intent columns,
# we will fill them with -1 so the loss function knows to ignore them (Masking).
intent_cols = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

for col in intent_cols:
    if col not in multi_train_df.columns:
        multi_train_df[col] = -1 # Special value for "No Label"

multi_labels = [
    {'toxic': t, 'intents': i.tolist()}
    for t, i in zip(multi_train_df['toxic'].values, multi_train_df[intent_cols].values)
]

multi_ds = TicketDataset(multi_train_df['text'].values, multi_labels, model_name="xlm-roberta-large")
multi_loader = DataLoader(multi_ds, batch_size=32, shuffle=True) # A100 handles this easily

In [16]:
# --- 1. Optimizer & Scheduler for Multilingual Pass ---
# We use a 5e-6 LR (lower than English) to avoid distorting the learned weights
multi_optimizer = AdamW(model.parameters(), lr=5e-6, weight_decay=0.01, fused=True)

# Total steps for 2 epochs of multilingual data
MULTI_EPOCHS = 2
total_multi_steps = len(multi_loader) * MULTI_EPOCHS

multi_scheduler = get_linear_schedule_with_warmup(
    multi_optimizer,
    num_warmup_steps=int(total_multi_steps * 0.1),
    num_training_steps=total_multi_steps
)

# --- 2. Execute Multilingual Training ---
print(f"Starting Multilingual Fine-Tuning on {device}...")

for epoch in range(MULTI_EPOCHS):
    print(f"\n--- Multilingual Epoch {epoch + 1}/{MULTI_EPOCHS} ---")

    # Use the train_epoch_multilingual function (ensure you've defined it from previous step!)
    avg_loss = train_epoch_multilingual(
        model,
        multi_loader,
        multi_optimizer,
        device,
        scaler,
        multi_scheduler
    )

    print(f"Epoch {epoch + 1} Avg Loss: {avg_loss:.4f}")

    # Save the "Global" Model
    torch.save(model.state_dict(), f"xlmr_large_global_v1_epoch{epoch+1}.bin")

Starting Multilingual Fine-Tuning on cuda...

--- Multilingual Epoch 1/2 ---


Multilingual Tuning:   0%|          | 0/1660 [00:00<?, ?it/s]/content/model_utils.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
Multilingual Tuning: 100%|██████████| 1660/1660 [03:24<00:00,  8.13it/s, loss=0.2448]


Epoch 1 Avg Loss: 0.3918

--- Multilingual Epoch 2/2 ---


Multilingual Tuning: 100%|██████████| 1660/1660 [03:23<00:00,  8.14it/s, loss=0.4066]


Epoch 2 Avg Loss: 0.2800


In [17]:
# 1. Load Multilingual Test Data
test_df = pd.read_csv("data/processed/multilingual_toxic/merged_non_en_test.csv").fillna("")

# 2. Setup Loader
# (Intents are [0]*5 here because we are only testing Toxicity on the non-English set)
test_labels = [{'toxic': t, 'intents': [0]*5} for t in test_df['toxic'].values]
test_ds = TicketDataset(test_df['text'].values, test_labels, model_name="xlm-roberta-large")
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

# 3. Load the NEW Multilingual Weights
# Using Epoch 2 as it was the most stable global checkpoint
model.load_state_dict(torch.load("xlmr_large_global_v1_epoch2.bin"))
model.to(device)
model.eval() # Ensure model is in eval mode

# 4. Run Evaluation
# Note: This now correctly captures the 4 return values
report, auc, lang_f1s, cm = ev.run_evaluation(
    model,
    test_loader,
    device,
    lang_list=test_df['language'].values
)

print("\n" + "="*50)
print("       FINAL GLOBAL MULTILINGUAL PERFORMANCE       ")
print("="*50)
print(report)
print(f"\nGLOBAL AUC-ROC: {auc:.4f}")

print("\nPER-LANGUAGE PERFORMANCE (Ascending Order):")
print("-" * 30)
# Sorting helps identify which languages might need more data in future work
for lang in sorted(lang_f1s, key=lang_f1s.get):
    print(f"{lang.upper():<12}: {lang_f1s[lang]:.4f}")

print("\nOVERALL CONFUSION MATRIX:")
print(cm)

Evaluating: 100%|██████████| 208/208 [00:13<00:00, 14.95it/s]



       FINAL GLOBAL MULTILINGUAL PERFORMANCE       
              precision    recall  f1-score   support

       Clean       0.87      0.87      0.87      6674
       Toxic       0.87      0.87      0.87      6601

    accuracy                           0.87     13275
   macro avg       0.87      0.87      0.87     13275
weighted avg       0.87      0.87      0.87     13275


GLOBAL AUC-ROC: 0.9464

PER-LANGUAGE PERFORMANCE (Ascending Order):
------------------------------
ZH          : 0.7571
HIN         : 0.7708
HE          : 0.8021
AR          : 0.8071
AM          : 0.8175
IT          : 0.8382
TT          : 0.8724
JA          : 0.8793
DE          : 0.8929
ES          : 0.8948
UK          : 0.9304
HI          : 0.9353
FR          : 0.9803
RU          : 0.9831

OVERALL CONFUSION MATRIX:
[[5830  844]
 [ 841 5760]]


In [9]:

en_test_df = pd.read_csv("data/processed/jigsaw/jigsaw_test.csv", low_memory=False)
print(en_test_df.columns)

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')


In [19]:

en_test_df = pd.read_csv("data/processed/jigsaw/jigsaw_test.csv", low_memory=False)
en_test_df = en_test_df[en_test_df['toxic'] != -1].sample(5000, random_state=42)
# 2. Prepare labels (Ensuring all 5 intent columns are present)
en_intent_cols = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
en_labels = [
    {'toxic': t, 'intents': i.tolist()}
    for t, i in zip(en_test_df['toxic'].values, en_test_df[en_intent_cols].values)
]

# 3. Create Dataset and Loader
en_test_ds = TicketDataset(en_test_df['comment_text'].values, en_labels, model_name="xlm-roberta-large")
en_test_loader = DataLoader(en_test_ds, batch_size=64, shuffle=False)

# 4. Run the Validation
print("Validating model on English Intent categories...")
val.validate_intents(model, en_test_loader, device)

Validating model on English Intent categories...


100%|██████████| 79/79 [00:05<00:00, 13.92it/s]


--- DEBUG: First 5 Rows (Post-Fix) ---
Sample 0 Labels: [nan nan nan nan nan]
Sample 0 Preds : [0.00047285 0.00970848 0.00106496 0.01322282 0.00288958]
Sample 1 Labels: [0. 0. 0. 0. 0.]
Sample 1 Preds : [0.00060708 0.02442309 0.00136703 0.02228619 0.0047552 ]
Sample 2 Labels: [nan nan nan nan nan]
Sample 2 Preds : [0.00041732 0.00124484 0.00050334 0.00116951 0.00068785]
Sample 3 Labels: [nan nan nan nan nan]
Sample 3 Preds : [0.00068785 0.00136703 0.0006667  0.00106496 0.00082959]
Sample 4 Labels: [0. 0. 0. 0. 0.]
Sample 4 Preds : [0.00060708 0.02442309 0.00136703 0.02228619 0.0047552 ]

--- FINAL PER-INTENT ROC-AUC ---
SEVERE_TOXIC    : 0.5000
OBSCENE         : 0.5000
THREAT          : 0.5000
INSULT          : 0.5000
IDENTITY_HATE   : 0.5000


# Inference of the model

In [20]:


# Must match the model name exactly to ensure the vocab IDs align
model_name = "xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.load_state_dict(torch.load("xlmr_large_multitask_epoch3.bin"))
model.to(device)
model.eval()

# 2. Hardcoded Toxic Sentence
test_text = "I am going to kill you, you disgusting idiot!" # Threat, Insult, Toxic
inputs = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    tox_logits, intent_logits = model(inputs['input_ids'], inputs['attention_mask'])

print(f"Raw Toxicity Logit: {tox_logits.item():.4f}")
print(f"Raw Intent Logits: {intent_logits.cpu().numpy()}")

# Convert to probabilities
print(f"Toxicity Prob: {torch.sigmoid(tox_logits).item():.4f}")
print(f"Intent Probs: {torch.sigmoid(intent_logits).cpu().numpy()}")

Raw Toxicity Logit: 5.9316
Raw Intent Logits: [[0.6768107  2.1690812  0.37225482 2.6320574  0.4047004 ]]
Toxicity Prob: 0.9974
Intent Probs: [[0.6630265  0.8974384  0.5920037  0.9328965  0.59981644]]
